# Portfolio review copy

**PORTFOLIO_REFACTORED_VERSION**  
This notebook was prepared from an existing team-project notebook for public review and reproducibility. It may differ from the file historically submitted or presented. Outputs and execution counts were removed, and local paths were changed to relative paths. No result was regenerated during this preparation.


# 페키지 로드 및 데이터 구조 확인

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


In [ ]:
from pathlib import Path

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir() and (candidate / "README.md").exists():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from the repository or its notebooks directory."
    )

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
stock_path = DATA_DIR / "주식통합.xlsx"
trend_path = DATA_DIR / "뉴스, 검색 데이터.xlsx"

print("주식통합:", stock_path)
print("뉴스/검색:", trend_path)

df_stock = pd.read_excel(stock_path)
df_trend = pd.read_excel(trend_path)

print("\n[주식통합 데이터 head]")
print(df_stock.head())
print("\n[주식통합 info]")
df_stock.info()
print("\n[뉴스/검색 데이터 head]")
print(df_trend.head())
print("\n[뉴스/검색 info]")
df_trend.info()


# 데이터 전처리 (날짜 변환, 컬럼명, Wide->long)

In [ ]:
# 주식 데이터 날짜 변환(object -> datetime64)
df_stock['date'] = pd.to_datetime(df_stock['일자'], errors='raise')

# 필요하면 원본 '일자'는 드롭
df_stock = df_stock.drop(columns=['일자'])

print(df_stock[['종목', 'date']].head())
df_stock.info()


In [ ]:
# 뉴스/검색 데이터 날짜 변환(object -> datetime64)
# -----------------------------
df_trend['date'] = pd.to_datetime(df_trend['일자'])

df_trend = df_trend.drop(columns=['일자'])

print(df_trend[['date']].head())
df_trend.info()


In [ ]:
# 검색량 컬럼 찾기 (문자열에 '검색량' 포함된 것)
search_cols = [c for c in df_trend.columns if '검색량' in c]

print("검색량 컬럼들:")
print(search_cols)

In [ ]:
# 뉴스 건수 컬럼 찾기
#  - 'NEWS 건수_' 로 시작하거나
#  - '건수_' 로 시작하는 컬럼 포함
# ------------------------------------
news_cols = [c for c in df_trend.columns 
             if c.startswith('NEWS 건수_') or c.startswith('건수_')]

print("뉴스 건수 컬럼들:")
print(news_cols)

In [ ]:
# 검색량 wide → long 변환
search_long = df_trend.melt(
    id_vars=['date'],           # 기준 축
    value_vars=search_cols,     # 펼칠 컬럼들
    var_name='검색컬럼명',
    value_name='검색량'
)

# 컬럼명에서 종목명 추출 (예: 'NAVER 검색량_펄어비스' → '펄어비스')
search_long['종목'] = search_long['검색컬럼명'].str.split('_').str[-1]

# 필요없는 원래 컬럼명은 제거
search_long = search_long.drop(columns=['검색컬럼명'])

print(search_long.head(10), search_long.tail(10))


In [ ]:
# 뉴스 건수 wide → long 변환
# ------------------------------------
news_long = df_trend.melt(
    id_vars=['date'],
    value_vars=news_cols,
    var_name='뉴스컬럼명',
    value_name='뉴스건수'
)

# 'NEWS 건수_삼성전자' → '삼성전자', '건수_셀트리온' → '셀트리온'
news_long['종목'] = news_long['뉴스컬럼명'].str.split('_').str[-1]

news_long = news_long.drop(columns=['뉴스컬럼명'])

print(news_long.head(10), search_long.tail(10))


In [ ]:
# 검색량 + 뉴스건수 병합 (date + 종목 기준)
# ------------------------------------
issue_long = pd.merge(
    search_long,
    news_long,
    on=['date', '종목'],
    how='outer'   # 어떤 쪽이 비어 있어도 다 살려두기
)

issue_long = issue_long[['date', '종목', '검색량', '뉴스건수']]

print(issue_long.head(20))
print(issue_long.info())


# 데이터 병합

In [ ]:
import numpy as np
import pandas as pd

# 2) 종목 필터: 이슈 데이터에 있는 종목만 사용
issue_stocks = issue_long['종목'].unique()
df_stock = df_stock[df_stock['종목'].isin(issue_stocks)].copy()

# 3) 날짜 범위도 이슈 데이터 범위 안으로 자르기
start_date = issue_long['date'].min()
end_date   = issue_long['date'].max()

df_stock = df_stock[
    (df_stock['date'] >= start_date) &
    (df_stock['date'] <= end_date)
].copy()

# 4) 정렬
df_stock = df_stock.sort_values(['종목', 'date']).reset_index(drop=True)

print("주식 종목:", sorted(df_stock['종목'].unique()))
print("주식 날짜 범위:", df_stock['date'].min(), "~", df_stock['date'].max())


In [ ]:
# 이슈 데이터도 정렬
df_issue = issue_long.sort_values(['종목', 'date']).reset_index(drop=True)

print(df_issue.head())
df_issue.info()

In [ ]:
# 이슈 기준으로 주식데이터 merge
df_all = df_issue.merge(
    df_stock,
    on=['종목', 'date'],
    how='left'   # 이슈 기준으로 모두 살리고, 주식 없는 날은 NaN
)

print(df_all.head(20))
print(df_all[['종목', 'date', '검색량', '뉴스건수', '종가', '거래량']].head(30))
df_all.info()

In [ ]:
# 변수 타임별로 ffill / 0 처리 분리 적용
# price/보유량(state) -> ffill(주말에도 상태 유지)
# price(상태) 계열 -> ffill금지, NaN이면 0처리
# 4-1. 컬럼 그룹 나누기
#----------------------------------------------------------
# price(상태) 계열
price_cols = ['종가', '시가', '고가', '저가']

# 보유량(상태) 계열
hold_cols = ['시가총액', '상장주식수', '외국인 보유수량', '외국인 지분율',
             '외국인 한도수량', '외국인 한도소진율']

# flow(흐름) 계열: 거래가 있는 날에만 의미가 있는 값들
flow_cols = ['거래량', '거래대금',
             '대비', '등락률',
             '개인(매도)', '개인(매수)', '개인(순매수)']


In [ ]:
# 4-3. price + holding만 ffill
# 종목별로 price/holding 상태를 주말까지 이어줌
df_all[price_cols + hold_cols] = (
    df_all
    .groupby('종목')[price_cols + hold_cols]
    .ffill()
)


In [ ]:
# 4-3. flow 계열은 ffill 금지 → NaN이면 0
df_all[flow_cols] = df_all[flow_cols].fillna(0)

In [ ]:
print(df_all.head(20))

In [ ]:
# 파생변수 : 이슈강도, 변동성, 로그 거래량

# 정렬 한 번 더 (안전하게)
df_all = df_all.sort_values(['종목', 'date']).reset_index(drop=True)

# 5-1. 이슈강도: 우선 뉴스건수 그대로 사용 (1차 버전)
df_all["뉴스_log"] = np.log1p(df_all["뉴스건수"])
df_all["검색_log"] = np.log1p(df_all["검색량"])

df_all["이슈강도"] = df_all["뉴스_log"] + df_all["검색_log"]



# 5-2. 변동성 지표: 등락률 절댓값
df_all['등락률_abs'] = df_all['등락률'].abs()

# 5-3. 거래량 로그변환 (0도 있을 수 있으니 log1p)
df_all['거래량_log'] = np.log1p(df_all['거래량'])

print("▶ 파생변수 예시")
print(df_all[['종목','date','이슈강도','등락률','등락률_abs','거래량','거래량_log']].head(20))


In [ ]:
# 6-1. 종목별 거래일 + next_trade_date는 그대로 OK
trade_dates = (
    df_stock[['종목','date']]
    .drop_duplicates()
    .sort_values(['종목','date'])
    .copy()
)
trade_dates['next_trade_date'] = trade_dates.groupby('종목')['date'].shift(-1)

print(trade_dates.head(10))


In [ ]:
# 혹시 모를 dtype 문제 대비
df_all['date'] = pd.to_datetime(df_all['date'])
trade_dates['date'] = pd.to_datetime(trade_dates['date'])

result_list = []

for stock, g in df_all.groupby('종목'):
    # 1) 이 종목의 이슈 타임라인 (주말 포함 가능)
    g = g.sort_values('date').reset_index(drop=True)
    
    # 2) 이 종목의 거래일 + next_trade_date
    td = trade_dates[trade_dates['종목'] == stock].copy()
    td = td.sort_values('date').reset_index(drop=True)
    
    # 3) asof를 위해 컬럼명 맞추기 (right의 키 이름 바꿔줌)
    td_renamed = td.rename(columns={'date': 'trade_date'})
    
    # 4) 이 종목의 각 날짜(date)에 대해
    #    - trade_date <= date 인 마지막 거래일을 찾고
    #    - 그 거래일의 next_trade_date를 붙임
    merged = pd.merge_asof(
        g,
        td_renamed[['trade_date', 'next_trade_date']],
        left_on='date',
        right_on='trade_date',
        direction='backward'
    )
    
    # trade_date 컬럼은 이제 필요 없으니 삭제
    merged = merged.drop(columns=['trade_date'])
    
    result_list.append(merged)

# 모든 종목 다시 합치기
df_all_with_next = (
    pd.concat(result_list, axis=0)
    .sort_values(['종목', 'date'])
    .reset_index(drop=True)
)

print("▶ 종목별 asof 이후 예시")
print(df_all_with_next[['종목','date','next_trade_date']].head(30))


In [ ]:
# 다음 거래일의 등락률 / 거래량_log 테이블 준비
future = (
    df_all_with_next[['종목','date','등락률','거래량_log']]
    .rename(columns={
        'date': 'next_trade_date',
        '등락률': '등락률_next',
        '거래량_log': '거래량_log_next',
    })
)

# next_trade_date 기준으로 merge
df_all_final = df_all_with_next.merge(
    future,
    on=['종목','next_trade_date'],
    how='left'
)

print("▶ 다음 거래일 반응 컬럼 확인")
print(
    df_all_final[
        ['종목','date','이슈강도','next_trade_date',
         '등락률','등락률_next','거래량_log','거래량_log_next']
    ].head(40)
)

# 결측치 확인 및 처리

In [ ]:
df_all_final.isna().sum()

`next_trade_date`, `등락률_next`, `거래량_log_next`는 다음 거래일의 값입니다.

입력 데이터의 마지막 관측일에는 다음 거래일이 존재하지 않아 이 값들이 결측일 수
있습니다. 해당 결측을 0으로 대체하면 실제 반응이 없었다고 오해할 수 있으므로
그대로 유지합니다. 공개 데이터가 없어 실제 마지막 관측일은 검증하지 않았습니다.

In [ ]:
OUTPUT_DIR = REPO_ROOT / "outputs" / "derived"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / "이슈기준_통합데이터_final.csv"
df_all_final.to_csv(output_path, index=False, encoding="utf-8-sig")
print("저장 완료:", output_path)


# 분석

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

def set_korean_font():
    candidates = (
        "Malgun Gothic",
        "NanumGothic",
        "Noto Sans CJK KR",
        "AppleGothic",
    )
    available = {font.name for font in font_manager.fontManager.ttflist}
    selected = next((font for font in candidates if font in available), None)
    if selected:
        plt.rcParams["font.family"] = selected
    plt.rcParams["axes.unicode_minus"] = False
    return selected

selected_font = set_korean_font()
print("Korean font:", selected_font or "viewer default")


# 1.클러스터링 분석

In [ ]:
# 종목별 이슈강도 Top1 날짜와 그 이후 첫 거래일 반응
top1_result = {}

for stock in df_all["종목"].unique():
    temp = (
        df_all[df_all["종목"] == stock]
        .sort_values("date")
        .copy()
    )

    top1 = (
        temp.groupby("date")["이슈강도"]
        .mean()
        .reset_index()
        .sort_values("이슈강도", ascending=False)
        .head(1)
    )
    if top1.empty:
        continue

    event_date = pd.to_datetime(top1.iloc[0]["date"])
    issue_score = top1.iloc[0]["이슈강도"]

    trading = temp[temp["거래량"].fillna(0) > 0].copy()
    response_row = trading[trading["date"] >= event_date].head(1)
    if response_row.empty:
        continue

    today = response_row.iloc[0]
    response_date = pd.to_datetime(today["date"])
    prev_trade = trading[trading["date"] < response_date].tail(1)

    if not prev_trade.empty:
        previous = prev_trade.iloc[0]
        rate_change = today["등락률"] - previous["등락률"]
        prev_volume = previous["거래량"]
        volume_change = (
            (today["거래량"] - prev_volume) / prev_volume * 100
            if pd.notna(prev_volume) and prev_volume != 0
            else np.nan
        )
        prev_date = previous["date"]
        prev_return = previous["등락률"]
    else:
        rate_change = np.nan
        volume_change = np.nan
        prev_date = pd.NaT
        prev_volume = np.nan
        prev_return = np.nan

    top1_result[stock] = {
        "이슈 Top1 날짜": event_date,
        "반응 거래일": response_date,
        "이슈강도": issue_score,
        "현재 등락률": today["등락률"],
        "직전거래일 등락률": prev_return,
        "전일대비 등락률 변화": rate_change,
        "현재 거래량": today["거래량"],
        "직전거래일": prev_date,
        "직전거래일 거래량": prev_volume,
        "전일대비 거래량 변화(%)": volume_change,
    }

top1_df = pd.DataFrame.from_dict(top1_result, orient="index")
print(top1_df)


In [ ]:
import numpy as np
import pandas as pd

# The original event date can be a non-trading day. For each stock, use the
# first observed trading row on or after that date, then take row-based
# windows so "±7" means trading observations rather than calendar days.
result_list = []

for stock in top1_df.index:
    event_date = pd.to_datetime(top1_df.loc[stock, "이슈 Top1 날짜"])
    temp = (
        df_all[df_all["종목"] == stock]
        .copy()
        .sort_values("date")
        .reset_index(drop=True)
    )
    temp["date"] = pd.to_datetime(temp["date"])
    temp = temp[temp["거래량"].fillna(0) > 0].reset_index(drop=True)

    eligible = temp.index[temp["date"] >= event_date].tolist()
    if not eligible:
        continue
    idx = eligible[0]

    start_idx = max(0, idx - 7)
    end_idx = min(len(temp) - 1, idx + 7)
    window = temp.iloc[start_idx : end_idx + 1].copy()

    ret_std = window["등락률"].std()
    past_20 = temp.iloc[max(0, idx - 20) : idx]
    past_volume = past_20["거래량"].mean()
    vol_spike = (
        temp.loc[idx, "거래량"] / past_volume
        if len(past_20) > 0 and pd.notna(past_volume) and past_volume > 0
        else np.nan
    )

    result_list.append([stock, ret_std, vol_spike])

cluster_df = pd.DataFrame(
    result_list,
    columns=["종목", "수익률_변동성", "거래량_스파이크"],
)


In [ ]:
# 결측 제거
cluster_df = cluster_df.dropna()

# 표준화
scaler = StandardScaler()
cluster_df[["ret_z", "vol_z"]] = scaler.fit_transform(
    cluster_df[["수익률_변동성", "거래량_스파이크"]]
)

# 민감도 점수 계산 (핵심!)
cluster_df["이슈_민감도"] = cluster_df["ret_z"] + cluster_df["vol_z"]


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# Sensitivity check on the same two standardized features used below.
# This diagnostic does not automatically select the historical K=3 setting.
X_cluster = cluster_df[["ret_z", "vol_z"]]
max_k = min(6, len(X_cluster) - 1)
if max_k < 2:
    raise ValueError("K sensitivity check requires at least three valid stock rows.")
K_range = range(2, max_k + 1)
inertias, sil_scores = [], []

for k in K_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_cluster)
    inertias.append(model.inertia_)
    sil_scores.append(silhouette_score(X_cluster, labels))

plt.figure(figsize=(8, 4))
plt.plot(list(K_range), inertias, marker="o")
plt.title("K-Means Sensitivity Check: Inertia")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(list(K_range), sil_scores, marker="o")
plt.title("K-Means Sensitivity Check: Silhouette")
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.grid(True)
plt.show()


In [ ]:
# Historical team setting used for the exploratory comparison.
FINAL_K = 3
if len(cluster_df) < FINAL_K:
    raise ValueError("K=3 requires at least three valid stock rows.")

kmeans = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10)
cluster_df["cluster"] = kmeans.fit_predict(
    cluster_df[["ret_z", "vol_z"]]
)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,7))
sns.scatterplot(
    data=cluster_df,
    x="ret_z", y="vol_z",
    hue="cluster", s=200
)

for i, row in cluster_df.iterrows():
    plt.text(row["ret_z"]+0.03, row["vol_z"]+0.03, row["종목"], fontsize=12)

plt.xlabel("수익률 변동성 (표준화)")
plt.ylabel("거래량 스파이크 (표준화)")
plt.title("이슈 민감도 기반 K-means 클러스터링")
plt.axhline(0, color="gray", linestyle="--")
plt.axvline(0, color="gray", linestyle="--")
plt.show()


# 2.종목별 시계열 흐름 

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ================================
# 1) 스파이크 감지 함수
# ================================
def detect_spike(series, window=20, std_k=2):
    s = series.fillna(0)
    roll_mean = s.rolling(window, min_periods=5).mean()
    roll_std  = s.rolling(window, min_periods=5).std()
    spike = (s > roll_mean + std_k * roll_std).astype(int)
    return spike.fillna(0)


# ================================
# 2) 시계열 시각화 (상위 퍼센타일 스파이크만 표시)
# ================================
def plot_flow_shading(df, stock, percentile=0.90):

    temp = df[df["종목"] == stock].sort_values("date").copy()
    temp = temp[["date", "검색량", "뉴스건수", "개인(순매수)", "종가"]].fillna(0)

    # ------ 스파이크 감지 ------
    temp["spike_search"] = detect_spike(temp["검색량"])
    temp["spike_price"] = detect_spike(temp["종가"])

    # 하나라도 spike이면 스파이크로 판단
    temp["spike"] = ((temp["spike_search"] == 1) | (temp["spike_price"] == 1)).astype(int)

    # ------ 정규화 ------
    scaler = MinMaxScaler()
    cols = ["검색량", "뉴스건수", "개인(순매수)", "종가"]
    temp[[c + "_s" for c in cols]] = scaler.fit_transform(temp[cols])

    # ------ 스파이크 중 상위 퍼센타일만 선택 ------
    spike_df = temp[temp["spike"] == 1].copy()
    if len(spike_df) > 0:
        threshold = spike_df["검색량_s"].quantile(percentile)
        strong_spike_df = spike_df[spike_df["검색량_s"] >= threshold]
    else:
        strong_spike_df = pd.DataFrame()

    # ------ 플롯 ------
    plt.figure(figsize=(20,6))

    plt.plot(temp["date"], temp["검색량_s"], label="검색량", color="blue", linewidth=1.8)
    plt.plot(temp["date"], temp["뉴스건수_s"], label="뉴스건수", color="orange", linewidth=1.8)
    plt.plot(temp["date"], temp["개인(순매수)_s"], label="개인 순매수", color="green", linewidth=1.8)
    plt.plot(temp["date"], temp["종가_s"], label="주가", color="red", linewidth=1.8)

    # ------ 강한 스파이크만 표시 ------
    for d in strong_spike_df["date"]:
        plt.axvspan(d, d + pd.Timedelta(days=1), color="red", alpha=0.15)

    for _, row in strong_spike_df.iterrows():
        x = row["date"]
        y = max(row["검색량_s"], row["종가_s"]) + 0.15

        plt.scatter(x, y, color="red", s=90, zorder=5)
        plt.text(x, y + 0.03, "SPK", fontsize=9, color="red", weight="bold")

    plt.title(f"[{stock}] 검색량·주가 스파이크 기반 시계열 흐름 (상위 퍼센타일 표시)", fontsize=17)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
for s in df_all_final["종목"].unique():
    plot_flow_shading(df_all_final, s)

이슈 반응

In [ ]:
# 이슈일 정의
# 해당 종목의 이슈강도 > 해당 종목의 rolling 평균 + 2 * 표준편차

In [ ]:
# 1) 종목별 rolling 평균 & 표준편차 계산
# -----------------------------------------------------
df_all['이슈_mean'] = df_all.groupby("종목")['이슈강도'].transform(lambda x: x.rolling(20, min_periods=5).mean())
df_all['이슈_std']  = df_all.groupby("종목")['이슈강도'].transform(lambda x: x.rolling(20, min_periods=5).std())

# -----------------------------------------------------
# 2) 이벤트일(Event Day) 정의
# -----------------------------------------------------
df_all['이벤트여부'] = (df_all['이슈강도'] > df_all['이슈_mean'] + 2 * df_all['이슈_std']).astype(int)

print("▶ 이벤트 발생 예시")
print(df_all[df_all['이벤트여부'] == 1].head(10)[['종목','date','이슈강도','이슈_mean','이슈_std']])

In [ ]:
# Event study with row-based trading-observation windows.
event_window = []

for stock, sub in df_all.groupby("종목"):
    sub = sub.copy().sort_values("date").reset_index(drop=True)
    sub["date"] = pd.to_datetime(sub["date"])
    event_dates = pd.to_datetime(
        sub.loc[sub["이벤트여부"] == 1, "date"].drop_duplicates()
    )
    trading_sub = (
        sub[sub["거래량"].fillna(0) > 0]
        .sort_values("date")
        .reset_index(drop=True)
    )

    for event_date in event_dates:
        eligible = trading_sub.index[trading_sub["date"] >= event_date].tolist()
        if not eligible:
            continue
        event_idx = eligible[0]
        event_trade_date = trading_sub.loc[event_idx, "date"]

        for lag in range(-3, 4):
            row_idx = event_idx + lag
            if 0 <= row_idx < len(trading_sub):
                row = trading_sub.iloc[row_idx]
                event_window.append(
                    {
                        "종목": stock,
                        "event_date": event_date,
                        "event_trade_date": event_trade_date,
                        "lag": lag,
                        "등락률": row["등락률"],
                        "거래량_log": row["거래량_log"],
                        "이슈강도": row["이슈강도"],
                    }
                )

event_df = pd.DataFrame(event_window)
print("이벤트 윈도우 예시")
print(event_df.head(15))


In [ ]:
# 4) lag별 평균 변화 계산
# -----------------------------------------------------
lag_result = event_df.groupby("lag")[["등락률", "거래량_log"]].mean().reset_index()

print("▶ lag별 평균 등락률/거래량 변화")
print(lag_result)

In [ ]:
# 등락률 이벤터 스터디 플롯
plt.figure(figsize=(10,5))
sns.lineplot(data=lag_result, x="lag", y="등락률", marker="o")
plt.axvline(0, color='red', linestyle='--')
plt.title("이슈 이벤트 전후 등락률 변화(±3일)")
plt.xlabel("이벤트 기준일 (lag)")
plt.ylabel("평균 등락률 (%)")
plt.grid(True)
plt.show()

# 거래량 이벤트 스터디 플롯
plt.figure(figsize=(10,5))
sns.lineplot(data=lag_result, x="lag", y="거래량_log", marker="o")
plt.axvline(0, color='red', linestyle='--')
plt.title("이슈 이벤트 전후 거래량 변화(±3일)")
plt.xlabel("이벤트 기준일 (lag)")
plt.ylabel("평균 거래량 log")
plt.grid(True)
plt.show()


## News-source excerpt review

The historical team notebook contained copied news excerpts and individual BIGKinds item links. They were removed from this public-review copy. Recreate and cite source evidence only through permitted access procedures.


## Public reproduction status

- `PORTFOLIO_REFACTORED_VERSION`
- `STATICALLY_VALIDATED`
- `CONDITIONAL_REPRODUCIBILITY`
- `FULL_RUN_NOT_VERIFIED`

Original and derived data are not included. Historical numeric interpretations remain unverified until a permitted dataset is used for a clean end-to-end run.